# AML Project 1 — Bank Marketing Dataset
## Missing Label Learning with Logistic Lasso Regression (FISTA)
**Advanced Machine Learning · 2026**

---
This notebook demonstrates the full pipeline:
- **Task 1** — Data preprocessing and missing label generation (MCAR / MAR1 / MAR2 / MNAR)
- **Task 2** — FISTA implementation and comparison with sklearn
- **Task 3** — UnlabeledLogReg with EM and Label Propagation

> **Files required in the same directory:**  
> `bank-additional-full.csv`, `bank_preprocessing.py`, `missing_data.py`, `fista.py`, `unlabeled_logreg.py`


## 0 · Imports & Setup

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
)
from sklearn.linear_model import LogisticRegression

# Project modules
from processing import run_pipeline
from missing_data_bank import generate_missing
from fista_bank import LogisticRegressionFISTA, FISTASelector
from unlabeled_logreg_bank import UnlabeledLogReg

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11
})

RANDOM_STATE = 42
C_MISSING    = 0.3
print("All imports OK ✓")


---
## Task 1 · Data Preparation and Missing Label Generation

### 1.1 — Preprocessing Pipeline

Steps applied by `bank_preprocessing.py`:
1. Column selection (pdays, default, housing, loan excluded)
2. Target encoding: `yes` → 1, `no` → 0
3. Feature engineering: `campaign_ratio = campaign / (previous + ε)`
4. One-hot encoding of 7 categorical features (`drop_first=True`)
5. Collinearity removal (|Pearson r| > 0.85)
6. Log1p transformation on skewed counts (`campaign`, `previous`)
7. Min-Max scaling → all numeric features in [0, 1]


In [1]:
df = run_pipeline("bank-additional-full.csv", "bank_preprocessed.csv")

print(f"Shape            : {df.shape}")
print(f"Target balance   : {df['y'].value_counts().to_dict()}")
print(f"Positive ratio   : {df['y'].mean():.3f}")
print(f"\nFirst 3 rows:")
df.head(3)


In [ ]:
# Feature overview
feature_names = [c for c in df.columns if c != "y"]
numeric_cols  = [c for c in feature_names if df[c].nunique() > 2]
dummy_cols    = [c for c in feature_names if df[c].nunique() <= 2]

print(f"Numeric features : {len(numeric_cols)}")
print(f"Dummy features   : {len(dummy_cols)}")
print(f"Total features   : {len(feature_names)}")
print(f"\nNumeric: {numeric_cols}")


In [ ]:
# Correlation heatmap of numeric features
import matplotlib.pyplot as plt

corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(numeric_cols, fontsize=9)
plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_title("Correlation matrix — numeric features")
plt.tight_layout()
plt.show()


In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

# Bar chart
counts = df["y"].value_counts().sort_index()
axes[0].bar(["No (0)", "Yes (1)"], counts.values, color=["#378ADD", "#1D9E75"], width=0.5)
axes[0].set_title("Class distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f"{v:,}", ha="center", fontsize=10)

# Age distribution by class
for label, color, name in [(0, "#378ADD", "No"), (1, "#D85A30", "Yes")]:
    subset = df[df["y"] == label]["age"]
    axes[1].hist(subset, bins=30, alpha=0.6, color=color, label=name, density=True)
axes[1].set_title("Age distribution by class (scaled)")
axes[1].set_xlabel("age (MinMax scaled)")
axes[1].legend()

plt.tight_layout()
plt.show()


### 1.2 — Train / Val / Test Split

In [ ]:
X = df[feature_names].values.astype(np.float64)
y = df["y"].values.astype(np.float64)

X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, stratify=y_tv, random_state=RANDOM_STATE
)

print(f"Train : {X_train.shape[0]:6,} rows  ({y_train.mean():.3f} positive)")
print(f"Val   : {X_val.shape[0]:6,} rows  ({y_val.mean():.3f} positive)")
print(f"Test  : {X_test.shape[0]:6,} rows  ({y_test.mean():.3f} positive)")


### 1.3 — Missing Label Generation

Four schemes applied to **training labels only** (test set always fully labeled).

| Scheme | Mechanism |
|--------|-----------|
| MCAR   | P(S=1) = c — purely random |
| MAR1   | P(S=1\|X) via sigmoid on `age` |
| MAR2   | P(S=1\|X) via random linear combination of all features |
| MNAR   | P(S=1\|X,Y) via `age` + true label Y |


In [ ]:
SCHEMES = {
    "MCAR": {"scheme": "mcar"},
    "MAR1": {"scheme": "mar1", "feature_idx": 0},        # age = col 0
    "MAR2": {"scheme": "mar2"},
    "MNAR": {"scheme": "mnar", "feature_idx": 0, "y_weight": 5.0},
}

print(f"Target missing rate c = {C_MISSING}")
print()
for name, kwargs in SCHEMES.items():
    y_obs = generate_missing(X_train, y_train, c=C_MISSING,
                             random_state=RANDOM_STATE, **kwargs)
    n_miss = (y_obs == -1).sum()
    n_obs  = (y_obs != -1).sum()
    pos_obs = (y_obs[y_obs != -1] == 1).sum()
    print(f"{name:5s}: missing={n_miss} ({n_miss/len(y_obs):.1%})  "
          f"observed={n_obs}  positives in observed={pos_obs}")


In [ ]:
# Visualise missingness pattern vs age for each scheme
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5), sharey=True)
age_col = X_train[:, 0]  # age is column 0

for ax, (name, kwargs) in zip(axes, SCHEMES.items()):
    y_obs = generate_missing(X_train, y_train, c=C_MISSING,
                             random_state=RANDOM_STATE, **kwargs)
    missing_mask = y_obs == -1
    ax.scatter(age_col[~missing_mask], y_train[~missing_mask],
               alpha=0.3, s=4, color="#378ADD", label="Observed")
    ax.scatter(age_col[missing_mask],  y_train[missing_mask],
               alpha=0.3, s=4, color="#E24B4A", label="Missing")
    ax.set_title(name)
    ax.set_xlabel("age (scaled)")

axes[0].set_ylabel("True label Y")
axes[-1].legend(loc="upper right", markerscale=3, fontsize=8)
fig.suptitle("Missingness pattern vs age by scheme", y=1.02)
plt.tight_layout()
plt.show()


---
## Task 2 · FISTA Implementation

### 2.1 — Algorithm Overview

FISTA minimises the Logistic Lasso objective:

$$\min_w \frac{1}{n}\sum_{i=1}^n \left[-y_i \log \sigma(w^\top x_i) - (1-y_i)\log(1-\sigma(w^\top x_i))\right] + \lambda\|w\|_1$$

**Key components:**
- **Lipschitz constant**: $L = \|X\|_F^2 / 4n$ → step size $1/L$
- **Soft thresholding** (proximal operator of L1): $S_\alpha(v) = \text{sign}(v)\cdot\max(|v|-\alpha, 0)$
- **Nesterov momentum**: $t_{k+1} = (1 + \sqrt{1 + 4t_k^2})/2$ → $O(1/k^2)$ convergence


### 2.2 — Lambda Selection

In [ ]:
print("Training FISTASelector (30 lambda values, roc_auc)...")
selector = FISTASelector(
    lambdas=np.logspace(-4, 1, 30),
    max_iter=1000,
    tol=1e-4
)
selector.fit(X_train, y_train, X_val, y_val, measure="roc_auc")

BEST_LAMBDA = selector.best_lambda
print(f"Best lambda: {BEST_LAMBDA:.6f}")
print(f"Best val ROC AUC: {selector.scores[BEST_LAMBDA]:.4f}")


In [ ]:
# Plot: validation ROC AUC vs lambda
selector.plot(measure="roc_auc")


In [ ]:
# Plot: regularisation path
selector.plot_coefficients(feature_names=feature_names)


### 2.3 — FISTA vs sklearn Comparison

In [ ]:
# Evaluate FISTA on test set
best_model = selector.best_model
fista_proba = best_model.predict_proba(X_test)
fista_pred  = (fista_proba >= 0.5).astype(int)

# sklearn L1 with equivalent regularisation strength
# sklearn uses C = 1/(n * lambda), penalty via l1_ratio=1.0
C_sk = 1.0 / (X_train.shape[0] * BEST_LAMBDA)
sk_model = LogisticRegression(
    solver="saga", C=C_sk, l1_ratio=1.0,
    max_iter=2000, random_state=RANDOM_STATE
)
sk_model.fit(X_train, y_train)
sk_proba = sk_model.predict_proba(X_test)[:, 1]
sk_pred  = sk_model.predict(X_test)

metrics = ["Recall", "Precision", "F1", "Balanced Accuracy", "ROC AUC", "PR AUC"]
from sklearn.metrics import recall_score, precision_score, average_precision_score

def all_metrics(y_true, y_pred, y_proba):
    return [
        recall_score(y_true, y_pred, zero_division=0),
        precision_score(y_true, y_pred, zero_division=0),
        f1_score(y_true, y_pred, zero_division=0),
        balanced_accuracy_score(y_true, y_pred),
        roc_auc_score(y_true, y_proba),
        average_precision_score(y_true, y_proba),
    ]

fista_scores  = all_metrics(y_test, fista_pred, fista_proba)
sklearn_scores = all_metrics(y_test, sk_pred, sk_proba)

results_t2 = pd.DataFrame({
    "Metric":  metrics,
    "FISTA (custom)": [f"{s:.4f}" for s in fista_scores],
    "sklearn L1":     [f"{s:.4f}" for s in sklearn_scores],
}).set_index("Metric")

print(f"Best lambda: {BEST_LAMBDA:.6f}")
print()
print(results_t2.to_string())


In [ ]:
# Coefficient comparison: FISTA vs sklearn
fista_coef  = best_model.w[1:]          # skip bias
sklearn_coef = sk_model.coef_[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Sorted absolute coefficients
idx = np.argsort(np.abs(fista_coef))[::-1][:20]
x = np.arange(len(idx))
w = 0.35
axes[0].bar(x - w/2, np.abs(fista_coef[idx]),  w, label="FISTA",   color="#378ADD", alpha=0.85)
axes[0].bar(x + w/2, np.abs(sklearn_coef[idx]), w, label="sklearn", color="#1D9E75", alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels([feature_names[i] for i in idx], rotation=45, ha="right", fontsize=8)
axes[0].set_title("Top 20 |coefficients| — FISTA vs sklearn")
axes[0].legend()

# Scatter
axes[1].scatter(sklearn_coef, fista_coef, alpha=0.6, s=20, color="#378ADD")
lim = max(np.abs(np.concatenate([fista_coef, sklearn_coef]))) * 1.1
axes[1].plot([-lim, lim], [-lim, lim], "r--", linewidth=1, label="y = x")
axes[1].set_xlabel("sklearn coefficient")
axes[1].set_ylabel("FISTA coefficient")
axes[1].set_title("Coefficient scatter: FISTA vs sklearn")
axes[1].legend()

plt.tight_layout()
plt.show()

n_zeros_fista   = (np.abs(fista_coef)  < 1e-6).sum()
n_zeros_sklearn = (np.abs(sklearn_coef) < 1e-6).sum()
print(f"Zero coefficients — FISTA: {n_zeros_fista}/{len(fista_coef)}  "
      f"sklearn: {n_zeros_sklearn}/{len(sklearn_coef)}")


---
## Task 3 · Unlabeled Logistic Regression

### 3.1 — Algorithm Overview

`UnlabeledLogReg` uses both labeled (Y ∈ {0,1}) and unlabeled (Y = −1) observations.

**Algorithm 1 — Label Propagation** (graph-based, one-shot):
1. Build k-NN graph over all observations; symmetrize
2. Convert distances to RBF weights: $w_{ij} = e^{-d_{ij}^2/\sigma^2}$
3. Row-normalize → transition matrix W
4. Solve harmonic system: $f_u = (I - W_{uu})^{-1} W_{ul} f_l$
5. Threshold at 0.5 → hard labels → fit FISTA

**Algorithm 2 — EM** (iterative):
1. Init: fit FISTA on labeled data → predict soft labels for unlabeled
2. M-step: refit FISTA on all observations using soft labels
3. E-step: update soft labels with new model predictions
4. Repeat until convergence (max |Δproba| < 1e-4)


### 3.2 — Experiment 1: Four Missing Schemes (c = 0.3)

In [ ]:
def run_all_schemes(method, lambda_val, c=0.3):
    """Run all 4 schemes and return results DataFrame."""
    model = UnlabeledLogReg(
        method=method,
        lambda_val=lambda_val,
        max_iter=20,
        fista_max_iter=1000,
        n_neighbors=15,
        random_state=RANDOM_STATE,
    )
    df_results = model.run_schemes(
        X_train, y_train, X_test, y_test,
        c=c, feature_idx=0, verbose=True
    )
    return df_results

print("=" * 60)
print("Method: EM")
print("=" * 60)
results_em = run_all_schemes("em", BEST_LAMBDA)


In [ ]:
print("=" * 60)
print("Method: Label Propagation")
print("=" * 60)
results_lp = run_all_schemes("label_propagation", BEST_LAMBDA)


In [ ]:
# Combined results table
print("\n=== EM Results ===")
print(results_em.to_string())
print("\n=== Label Propagation Results ===")
print(results_lp.to_string())


In [ ]:
# Visualise: ROC AUC and F1 by scheme and method
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
schemes = ["MCAR", "MAR1", "MAR2", "MNAR"]
x = np.arange(len(schemes))
w = 0.2

colors = {"Naive": "#378ADD", "EM": "#1D9E75", "LP": "#D85A30", "Oracle": "#888780"}
method_labels = ["Naive", "EM", "LP", "Oracle"]

for ax_idx, metric in enumerate(["ROC AUC", "F1"]):
    ax = axes[ax_idx]
    
    # EM results
    naive_vals  = [results_em.loc[(s, "Naive"),  metric] for s in schemes]
    em_vals     = [results_em.loc[(s, "EM"),     metric] for s in schemes]
    oracle_vals = [results_em.loc[(s, "Oracle"), metric] for s in schemes]
    
    # LP results
    lp_vals = [results_lp.loc[(s, "LABEL_PROPAGATION"), metric] for s in schemes]

    ax.bar(x - 1.5*w, naive_vals,  w, label="Naive",  color=colors["Naive"],  alpha=0.85)
    ax.bar(x - 0.5*w, em_vals,     w, label="EM",     color=colors["EM"],     alpha=0.85)
    ax.bar(x + 0.5*w, lp_vals,     w, label="LP",     color=colors["LP"],     alpha=0.85)
    ax.bar(x + 1.5*w, oracle_vals, w, label="Oracle", color=colors["Oracle"], alpha=0.85)
    
    ax.set_xticks(x)
    ax.set_xticklabels(schemes)
    ax.set_title(f"{metric} by scheme and method")
    ax.set_ylabel(metric)
    ax.legend(fontsize=9)
    
    # Highlight MNAR
    ax.axvspan(2.5, 3.5, alpha=0.08, color="red", label="_nolegend_")
    ax.text(3, ax.get_ylim()[0] + 0.01, "MNAR", ha="center", fontsize=9, color="red")

plt.tight_layout()
plt.show()


In [ ]:
# ROC curve comparison — MCAR scheme
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)

for ax, (sname, kwargs) in zip(axes, SCHEMES.items()):
    y_obs = generate_missing(X_train, y_train, c=C_MISSING,
                             random_state=RANDOM_STATE, **kwargs)
    
    # Fit all models
    em_m = UnlabeledLogReg("em",     lambda_val=BEST_LAMBDA, max_iter=20,
                           fista_max_iter=500, random_state=RANDOM_STATE)
    lp_m = UnlabeledLogReg("label_propagation", lambda_val=BEST_LAMBDA, max_iter=20,
                           fista_max_iter=500, n_neighbors=15, random_state=RANDOM_STATE)
    em_m.fit(X_train, y_obs);   em_m.naive_fit(X_train, y_obs)
    em_m.oracle_fit(X_train, y_train)
    lp_m.fit(X_train, y_obs)

    models_probas = {
        "Naive":  em_m.model_naive_.predict_proba(X_test),
        "EM":     em_m.predict_proba(X_test),
        "LP":     lp_m.predict_proba(X_test),
        "Oracle": em_m.model_oracle_.predict_proba(X_test),
    }
    plot_colors = ["#378ADD", "#1D9E75", "#D85A30", "#888780"]
    
    for (name, proba), color in zip(models_probas.items(), plot_colors):
        fpr, tpr, _ = roc_curve(y_test, proba)
        auc = roc_auc_score(y_test, proba)
        ax.plot(fpr, tpr, color=color, linewidth=1.5, label=f"{name} ({auc:.3f})")
    
    ax.plot([0,1],[0,1],"k--", linewidth=0.8)
    ax.set_title(sname)
    ax.set_xlabel("FPR")
    ax.legend(fontsize=7)

axes[0].set_ylabel("TPR")
fig.suptitle("ROC curves by missing scheme (c=0.3)", y=1.02)
plt.tight_layout()
plt.show()


### 3.3 — Experiment 2: MCAR Sensitivity (varying c)

In [ ]:
print("Running MCAR sensitivity analysis...")
model_sens = UnlabeledLogReg(
    method="em",
    lambda_val=BEST_LAMBDA,
    max_iter=20,
    fista_max_iter=1000,
    random_state=RANDOM_STATE,
)
sens_results = model_sens.run_mcar_sensitivity(
    X_train, y_train, X_test, y_test,
    c_values=[0.1, 0.2, 0.3, 0.4, 0.5],
    verbose=False
)
print(sens_results.to_string())


In [ ]:
# Plot sensitivity
c_vals = [0.1, 0.2, 0.3, 0.4, 0.5]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax_idx, metric in enumerate(["ROC AUC", "F1"]):
    ax = axes[ax_idx]
    for method_name, color, ls in [
        ("Naive",  "#378ADD", "-o"),
        ("EM",     "#1D9E75", "-s"),
        ("Oracle", "#888780", "--^"),
    ]:
        try:
            vals = [sens_results.loc[(c, method_name), metric] for c in c_vals]
        except KeyError:
            continue
        ax.plot(c_vals, vals, ls, color=color, linewidth=2, markersize=6, label=method_name)
    
    ax.set_xlabel("Missing rate c (MCAR)")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} vs c — MCAR setting")
    ax.set_xticks(c_vals)
    ax.legend()

plt.tight_layout()
plt.show()


### 3.4 — Key Findings

| Scheme | Finding |
|--------|---------|
| **MCAR** | All methods perform near-identically (ROC AUC gap < 0.003). Naive sufficient — no benefit from using unlabeled data when missingness is truly random. |
| **MAR1/MAR2** | Naive outperforms EM and LP. Systematic missingness biases the unlabeled data; EM reinforces this bias iteratively. LP propagates the skew through the graph. |
| **MNAR** | All methods collapse to F1 = 0.000, Balanced Accuracy = 0.500. Positive clients (Y=1) disproportionately missing → model predicts all-negative. EM worsens ROC AUC due to confirmation bias. |
| **c sensitivity** | Performance degrades only slightly as c increases 0.1→0.5 under MCAR. With 24,712 training samples, even 50% missingness leaves sufficient labeled data for logistic regression. |


---
## Summary

In [ ]:
# Full results summary
print("\n" + "=" * 70)
print("FULL EXPERIMENT RESULTS — EM")
print("=" * 70)
print(results_em.to_string())

print("\n" + "=" * 70)
print("FULL EXPERIMENT RESULTS — Label Propagation")
print("=" * 70)
print(results_lp.to_string())

print("\n" + "=" * 70)
print("TASK 2 — FISTA vs sklearn")
print("=" * 70)
print(results_t2.to_string())
